# CIGRE MV14 local-subsystem $A$ matrix and eigenvalues

This notebook instantiates the reduced current-coupling matrix derived for the GFL-GFM subsystem using parameters from `CIGREMV14System`.

The local three-branch reduction is centered at **bus 8**:

- grid path: source-1-2-3-8;
- GFL path: branch 7-8;
- GFM path: branch 8-9 plus the GFM virtual impedance.

All network parameters are already on the 25 MVA system base. The GFM virtual resistance and inductance are converted from the 5 MVA converter base to the system base. The resulting matrix represents the four current states $\left[\Delta i_{Ld},\Delta i_{Lq},\Delta i_{Md},\Delta i_{Mq}\right]^T$; it is a local high-frequency reduction, not the complete 94-state CIGRE Jacobian.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.linalg import block_diag
from scipy.optimize import linear_sum_assignment

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "codesimpy").is_dir())
sys.path.insert(0, str(ROOT))

from codesimpy.systems.cigre_mv14_system import CIGREMV14System

np.set_printoptions(precision=8, suppress=True)

## 1. Extract the numerical parameters

At the same voltage base, a per-unit impedance changes base according to

$$
Z_{\mathrm{pu,new}}
=
Z_{\mathrm{pu,old}}
\frac{S_{\mathrm{new}}}{S_{\mathrm{old}}}.
$$

The CIGRE implementation sets $R_v=0.1$ and $L_v=0.3$ on the 5 MVA GFM base. The code below converts them to the 25 MVA network base.

In [2]:
p = CIGREMV14System.defaultParameters()
branch_rows = CIGREMV14System.buildCigreData()["branchTable"]
branches = {(start, stop): dict(R=R, L=L, C=C)
            for start, stop, R, L, C in branch_rows}

grid_path = [("source", "1"), ("1", "2"), ("2", "3"), ("3", "8")]
Rg = sum(branches[edge]["R"] for edge in grid_path)
Lg = sum(branches[edge]["L"] for edge in grid_path)

R2, L2 = branches[("7", "8")]["R"], branches[("7", "8")]["L"]
R1, L1 = branches[("8", "9")]["R"], branches[("8", "9")]["L"]

gfm_base_ratio = p["Sbase_VA"] / p["VSGSbase_VA"]
Rv = 0.1 * gfm_base_ratio
Lv = 0.3 * gfm_base_ratio
LM = L1 + Lv

parameter_table = pd.DataFrame(
    [
        ("Grid equivalent", "source–1–2–3–8", Rg, Lg),
        ("GFL branch", "7–8", R2, L2),
        ("GFM physical branch", "8–9", R1, L1),
        ("GFM virtual impedance", "device, converted base", Rv, Lv),
        ("GFM combined branch", "physical + virtual", R1 + Rv, LM),
    ],
    columns=["Subsystem", "CIGRE location", "R (p.u.)", "L (p.u.)"],
)
display(parameter_table.style.format({"R (p.u.)": "{:.8f}", "L (p.u.)": "{:.8f}"}))
print(f"GFM base-conversion factor: {gfm_base_ratio:.1f}")

,Subsystem,CIGRE location,R (p.u.),L (p.u.)
0,Grid equivalent,source–1–2–3–8,0.26890627,0.50714019
1,GFL branch,7–8,0.05229188,0.07473250
2,GFM physical branch,8–9,0.01002000,0.01432000
3,GFM virtual impedance,"device, converted base",0.50000000,1.50000000
4,GFM combined branch,physical + virtual,0.51002000,1.51432000


GFM base-conversion factor: 5.0


## 2. Construct the coupled matrix

In the synchronous $dq$ frame, define

$$
J=\begin{bmatrix}0&-1\\1&0\end{bmatrix},
\qquad
Z_k=R_kI_2+\omega_0L_kJ,
\qquad
\omega_0=1\ \mathrm{p.u.},
$$

and

$$
Z_M=(R_1+R_v)I_2+\omega_0(L_1+L_v)J.
$$

For $x=[\Delta i_L^T,\Delta i_M^T]^T$,

$$
\mathcal M\dot x=-\mathcal Kx,
$$

$$
\mathcal M=
\begin{bmatrix}
(L_g+L_2)I_2&L_gI_2\\
L_gI_2&(L_g+L_M)I_2
\end{bmatrix},
\qquad
\mathcal K=
\begin{bmatrix}
Z_g+Z_2&Z_g\\
Z_g&Z_g+Z_M
\end{bmatrix}.
$$

Hence $A_\tau=-\mathcal M^{-1}\mathcal K$. CodeSim uses normalized time $\tau=\omega_b t$, so the matrix in $\mathrm{s}^{-1}$ is $A=\omega_bA_\tau$.

In [3]:
I2 = np.eye(2)
J = np.array([[0.0, -1.0], [1.0, 0.0]])
omega0_pu = 1.0
omega_base = 2 * np.pi * p["frequency_Hz"]

Zg = Rg * I2 + omega0_pu * Lg * J
Z2 = R2 * I2 + omega0_pu * L2 * J
ZM = (R1 + Rv) * I2 + omega0_pu * LM * J

M = np.block([
    [(Lg + L2) * I2, Lg * I2],
    [Lg * I2, (Lg + LM) * I2],
])
K = np.block([
    [Zg + Z2, Zg],
    [Zg, Zg + ZM],
])

A_tau = -np.linalg.solve(M, K)
A = omega_base * A_tau
state_names = ["ΔiLd", "ΔiLq", "ΔiMd", "ΔiMq"]

# Independent check against the explicit block formula.
a, c = Lg + L2, Lg + LM
D = a * c - Lg**2
A_tau_explicit = np.block([
    [-(LM * Zg + c * Z2) / D, (Lg * ZM - LM * Zg) / D],
    [(Lg * Z2 - L2 * Zg) / D, -(L2 * Zg + a * ZM) / D],
])
np.testing.assert_allclose(A_tau, A_tau_explicit, rtol=1e-12, atol=1e-12)

print("Aτ, normalized-time matrix:")
display(pd.DataFrame(A_tau, index=state_names, columns=state_names)
        .style.format("{:.8f}"))
print("A = ωb Aτ, in s⁻¹:")
display(pd.DataFrame(A, index=state_names, columns=state_names)
        .style.format("{:.4f}"))

Aτ, normalized-time matrix:


,ΔiLd,ΔiLq,ΔiMd,ΔiMq
ΔiLd,-0.55809906,1.00000000,-0.16164508,-0.00000000
ΔiLq,-1.00000000,-0.55809906,-0.00000000,-0.16164508
ΔiMd,0.00698910,-0.00000000,-0.34477531,1.00000000
ΔiMq,-0.00000000,0.00698910,-1.00000000,-0.34477531


A = ωb Aτ, in s⁻¹:


,ΔiLd,ΔiLq,ΔiMd,ΔiMq
ΔiLd,-175.3320,314.1593,-50.7823,-0.0000
ΔiLq,-314.1593,-175.3320,-0.0000,-50.7823
ΔiMd,2.1957,-0.0000,-108.3144,314.1593
ΔiMq,-0.0000,2.1957,-314.1593,-108.3144


## 3. Local-subsystem modal transformation

Partition

$$
A=
\begin{bmatrix}
A_{LL}&A_{LM}\\
A_{ML}&A_{MM}
\end{bmatrix}.
$$

Diagonalize the isolated local blocks:

$$
A_{LL}=T_L\Lambda_LT_L^{-1},
\qquad
A_{MM}=T_M\Lambda_MT_M^{-1}.
$$

The block-local transformation is

$$
T_{\mathrm{loc}}=\operatorname{blkdiag}(T_L,T_M),
$$

which gives

$$
A_{\mathrm{loc}}
=
T_{\mathrm{loc}}^{-1}AT_{\mathrm{loc}}
=
\begin{bmatrix}
\Lambda_L&C_{LM}\\
C_{ML}&\Lambda_M
\end{bmatrix}.
$$

The diagonal entries are the **local subsystem eigenvalues**. The eigenvalues of the complete transformed matrix are the **final coupled eigenvalues**. Similarity guarantees $\operatorname{eig}(A_{\mathrm{loc}})=\operatorname{eig}(A)$.

In [4]:
A_LL, A_MM = A[:2, :2], A[2:, 2:]
lambda_L, T_L = np.linalg.eig(A_LL)
lambda_M, T_M = np.linalg.eig(A_MM)

T_local = block_diag(T_L, T_M)
A_local = np.linalg.solve(T_local, A @ T_local)

local_eigenvalues = np.r_[lambda_L, lambda_M]
local_labels = np.array(["GFL local"] * 2 + ["GFM local"] * 2)
final_eigenvalues = np.linalg.eigvals(A_local)
direct_eigenvalues = np.linalg.eigvals(A)

np.testing.assert_allclose(
    np.sort_complex(final_eigenvalues),
    np.sort_complex(direct_eigenvalues),
    rtol=1e-11, atol=1e-11,
)
np.testing.assert_allclose(
    A_local[:2, :2], np.diag(lambda_L), rtol=1e-11, atol=1e-11
)
np.testing.assert_allclose(
    A_local[2:, 2:], np.diag(lambda_M), rtol=1e-11, atol=1e-11
)

# One-to-one matching is used only to draw local-to-final shift lines.
row_index, column_index = linear_sum_assignment(
    np.abs(local_eigenvalues[:, None] - final_eigenvalues[None, :])
)
matched_final = np.empty_like(local_eigenvalues)
matched_final[row_index] = final_eigenvalues[column_index]

def mode_rows(values, labels):
    rows = []
    for label, value in zip(labels, values):
        rows.append({
            "Mode": label,
            "Real (s⁻¹)": value.real,
            "Imaginary (s⁻¹)": value.imag,
            "Frequency (Hz)": abs(value.imag) / (2 * np.pi),
            "Damping ratio": -value.real / max(abs(value), np.finfo(float).eps),
        })
    return rows

mode_table = pd.DataFrame(
    mode_rows(local_eigenvalues, local_labels)
    + mode_rows(final_eigenvalues, ["Final coupled"] * len(final_eigenvalues))
)
display(mode_table.style.format({
    "Real (s⁻¹)": "{:.4f}",
    "Imaginary (s⁻¹)": "{:.4f}",
    "Frequency (Hz)": "{:.4f}",
    "Damping ratio": "{:.4f}",
}))

print("A_local = T_local⁻¹ A T_local:")
display(pd.DataFrame(
    A_local,
    index=["GFL λ₁", "GFL λ₂", "GFM λ₁", "GFM λ₂"],
    columns=["GFL λ₁", "GFL λ₂", "GFM λ₁", "GFM λ₂"],
).style.format(lambda z: f"{z.real:.3f}{z.imag:+.3f}j"))

print("Maximum final-eigenvalue consistency error:",
      np.max(np.abs(np.sort_complex(final_eigenvalues)
                    - np.sort_complex(direct_eigenvalues))))

,Mode,Real (s⁻¹),Imaginary (s⁻¹),Frequency (Hz),Damping ratio
0,GFL local,-175.3320,314.1593,50.0000,0.4873
1,GFL local,-175.3320,-314.1593,50.0000,0.4873
2,GFM local,-108.3144,314.1593,50.0000,0.3259
3,GFM local,-108.3144,-314.1593,50.0000,0.3259
4,Final coupled,-173.6247,314.1593,50.0000,0.4837
5,Final coupled,-110.0216,314.1593,50.0000,0.3305
6,Final coupled,-173.6247,-314.1593,50.0000,0.4837
7,Final coupled,-110.0216,-314.1593,50.0000,0.3305


A_local = T_local⁻¹ A T_local:


,GFL λ₁,GFL λ₂,GFM λ₁,GFM λ₂
GFL λ₁,-175.332+314.159j,0.000+0.000j,-50.782+0.000j,-0.000+0.000j
GFL λ₂,0.000-0.000j,-175.332-314.159j,0.000+0.000j,-50.782-0.000j
GFM λ₁,2.196+0.000j,0.000+0.000j,-108.314+314.159j,-0.000+0.000j
GFM λ₂,-0.000-0.000j,2.196+0.000j,0.000-0.000j,-108.314-314.159j


Maximum final-eigenvalue consistency error: 2.608766145426361e-13


In [8]:
FONT = dict(family="Times New Roman", size=22, color="black")
fig = go.Figure()

for local_value, final_value in zip(local_eigenvalues, matched_final):
    fig.add_trace(go.Scatter(
        x=[local_value.real, final_value.real],
        y=[local_value.imag, final_value.imag],
        mode="lines",
        line=dict(color="#8C8C8C", width=1.1, dash="dot"),
        hoverinfo="skip",
        showlegend=False,
    ))

gfl = local_labels == "GFL local"
gfm = local_labels == "GFM local"
fig.add_trace(go.Scatter(
    x=local_eigenvalues[gfl].real,
    y=local_eigenvalues[gfl].imag,
    mode="markers",
    name="GFL local eigenvalues",
    marker=dict(symbol="circle-open", size=15, color="blue", line_width=2.2),
))
fig.add_trace(go.Scatter(
    x=local_eigenvalues[gfm].real,
    y=local_eigenvalues[gfm].imag,
    mode="markers",
    name="GFM local eigenvalues",
    marker=dict(symbol="diamond-open", size=15, color="green", line_width=2.2),
))
fig.add_trace(go.Scatter(
    x=final_eigenvalues.real,
    y=final_eigenvalues.imag,
    mode="markers",
    name="Final coupled eigenvalues",
    marker=dict(symbol="x", size=15, color="#A2142F", line_width=2.5),
))

axis = dict(
    showline=True, linecolor="black", linewidth=1, mirror=True,
    ticks="inside", ticklen=4, tickfont=FONT, title_font=FONT,
    showgrid=True, gridcolor="#E6E6E6", gridwidth=0.5, zeroline=False,
)
fig.update_xaxes(title_text="Real part (s⁻¹)", **axis)
fig.update_yaxes(title_text="Imaginary part (s⁻¹)", **axis)
fig.add_vline(x=0, line=dict(color="black", width=1))
fig.add_hline(y=0, line=dict(color="black", width=1))
fig.update_layout(
    template="plotly_white", width=700, height=500, font=FONT,
    # title=dict(
    #     text="CIGRE MV14 local and final eigenvalues",
    #     x=0.5, xanchor="center", font=FONT,
    # ),
    margin=dict(l=95, r=30, t=95, b=80),
    legend=dict(
        orientation="h", x=0.5, xanchor="center",
        y=1.02, yanchor="bottom", font=FONT,
        bgcolor="rgba(255,255,255,0)",
    ),
)
fig.show(config={"displaylogo": False})

## Interpretation

The dotted lines show the displacement from each isolated local mode to its nearest final coupled mode. In this reduction, all modes remain stable. The imaginary parts stay near the 50 Hz synchronous frequency, while the shared grid path shifts the modal damping represented by the real parts.

The transformed matrix retains nonzero off-diagonal interaction blocks $C_{LM}$ and $C_{ML}$. They are the modal-coordinate representation of coupling through the shared bus-8 voltage and the grid current.